In [1]:
train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


NameError: name 'pd' is not defined

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error

In [ ]:
TARGET = "TargetValue"
ID_COLUMN = "TransactionID"

test_ids = test[ID_COLUMN].copy()

all_features = pd.concat(
    [train.drop(columns=[TARGET]), test],
    axis=0,
    ignore_index=True
)

date = pd.to_datetime(all_features["TransactionDate"], errors="coerce")

all_features["TransactionYear"] = date.dt.year
all_features["TransactionMonth"] = date.dt.month
all_features["TransactionQuarter"] = date.dt.quarter
all_features["TransactionDayOfYear"] = date.dt.dayofyear
all_features["TransactionDays"] = (date - pd.Timestamp("1990-01-01")).dt.days

all_features["ManufactureYear"] = pd.to_numeric(
    all_features["ManufactureYear"],
    errors="coerce"
)

all_features["ManufactureYear"] = all_features["ManufactureYear"].where(
    all_features["ManufactureYear"].between(1900, 2020)
)

all_features["AssetAge"] = all_features["TransactionYear"] - all_features["ManufactureYear"]
all_features["AssetAge"] = all_features["AssetAge"].where(
    all_features["AssetAge"].between(0, 100)
)

all_features["LogOperationalHours"] = np.log1p(
    pd.to_numeric(all_features["OperationalHoursMeter"], errors="coerce").clip(lower=0)
)

for col in [
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "Spec_SubClass",
    "VendorPartnerID",
    "RegionCode"
]:
    all_features[f"{col}_Frequency"] = all_features[col].map(
        all_features[col].value_counts(dropna=False)
    ).astype("float32")

all_features = all_features.drop(
    columns=["AssetID", ID_COLUMN, "TransactionDate"],
    errors="ignore"
)

X = all_features.iloc[:len(train)].copy()
X_test = all_features.iloc[len(train):].copy()
y = np.log1p(train[TARGET])

target_encode_cols = [
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "Spec_SubClass",
    "VendorPartnerID",
    "RegionCode",
    "InventoryGroupCategory",
    "InventoryGroupDescription"
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_encode_cols:
    if col not in X.columns:
        continue

    X[f"{col}_TE"] = np.nan
    X_test[f"{col}_TE"] = 0.0

    for train_idx, valid_idx in kf.split(X):
        stats = y.iloc[train_idx].groupby(X.iloc[train_idx][col]).mean()
        X.loc[X.index[valid_idx], f"{col}_TE"] = X.iloc[valid_idx][col].map(stats)

    full_stats = y.groupby(X[col]).mean()
    X_test[f"{col}_TE"] = X_test[col].map(full_stats)

    global_mean = y.mean()
    X[f"{col}_TE"] = X[f"{col}_TE"].fillna(global_mean).astype("float32")
    X_test[f"{col}_TE"] = X_test[f"{col}_TE"].fillna(global_mean).astype("float32")

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

if "ProductConfigID" in X.columns:
    categorical_columns.append("ProductConfigID")

categorical_columns = list(dict.fromkeys(categorical_columns))

for col in categorical_columns:
    X[col] = X[col].astype(str).fillna("__MISSING__").astype("category")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__").astype("category")

params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 6000,
    "learning_rate": 0.025,
    "num_leaves": 95,
    "min_child_samples": 30,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.05,
    "reg_lambda": 1.5,
    "cat_smooth": 30,
    "cat_l2": 15,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True
}

valid_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_columns,
        eval_set=[(X_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(200)
        ]
    )

    valid_predictions[valid_idx] = np.expm1(
        model.predict(X_valid, num_iteration=model.best_iteration_)
    )

    test_predictions += np.expm1(
        model.predict(X_test, num_iteration=model.best_iteration_)
    ) / kf.n_splits

    best_iterations.append(model.best_iteration_)

    fold_rmsle = np.sqrt(
        mean_squared_log_error(
            np.expm1(y_valid),
            np.clip(valid_predictions[valid_idx], 0, None)
        )
    )

    print(f"Fold {fold} RMSLE: {fold_rmsle:.6f}")

valid_predictions = np.clip(valid_predictions, 0, None)
test_predictions = np.clip(test_predictions, 0, None)

cv_rmsle = np.sqrt(
    mean_squared_log_error(
        train[TARGET],
        valid_predictions
    )
)

print("CV RMSLE:", cv_rmsle)
print("Average best iteration:", int(np.mean(best_iterations)))

submission[ID_COLUMN] = test_ids
submission[TARGET] = test_predictions

submission.to_csv("submission.csv", index=False)

print("Created submission.csv")
print(submission.head())

# MILESTONE-1

In [ ]:
print(train.shape,test.shape)

In [ ]:
set(train.columns)-set(test.columns)

In [ ]:
cols=train.select_dtypes(include='number').columns
len(cols)


In [ ]:
missing=train.isna().sum()
missing[missing > 124800]

In [ ]:
train['OperationalHoursMeter'].isna().mean()*100

In [ ]:
train['TargetValue'].median()

In [ ]:
train['ManufactureYear'].mode()[0]

In [ ]:
train['TransactionDate']=pd.to_datetime(train['TransactionDate'])
train['TransactionDate'].min()

In [ ]:
train['Spec_BaseClass'].nunique()

In [ ]:
train['RegionCode'].value_counts().head()

In [ ]:
train[train['RegionCode']=='Florida']['TargetValue'].mean()

In [ ]:
train['UtilizationTier'].value_counts(dropna=False)

In [ ]:
train[['TargetValue','OperationalHoursMeter']].corr()

In [ ]:
train['FunctionalClassification'].value_counts().head()